In [0]:
%run /Configurations/Init_Scripts

In [0]:
dbutils.widgets.text('startingVersion','0')
startingVersion = dbutils.widgets.get('startingVersion')
print("startingVersion:"+str(startingVersion))

dbutils.widgets.text("PromotionName", "4P Beta")
PromotionName = dbutils.widgets.get("PromotionName")

dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","49")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','12')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('CreatedBy','ADB_BillingCase')
CreatedBy = dbutils.widgets.get('CreatedBy')

In [0]:
# Source path:
checkPointLocation = "/_checkpoints/"
silverpath_InvoiceAddendum = '/mnt/silver/FACTInvoiceAddendum'
rawpath_BillingCase = '/mnt/raw/SalesForce/Moxie/BillingCase'


In [0]:
df_Source = (spark.readStream.option("cloudFiles.maxFilesPerTrigger",5000) 
                              .option("cloudFiles.maxBytesPerTrigger",'10g')
                               .option("startingVersion", startingVersion)
                               .option("skipChangeCommits", "true")
                               .format("delta").load(rawpath_BillingCase))

In [0]:
def upsertToDelta(microBatchOutputDF, batchId):     
    try:
        
        print("Running for BatchID:"+str(batchId))

        df_BillingCase = microBatchOutputDF.withColumn("DropDuplicates",row_number().over(Window.partitionBy("MoxieCaseID").orderBy(col("UpdatedDate").desc()))).filter("DropDuplicates = 1")

        targetDF = DeltaTable.forPath(spark, silverpath_InvoiceAddendum)  
        (targetDF.alias("tgt")
        .merge(df_BillingCase.alias("src"), "src.MoxieCaseID = tgt.MoxieCaseID ")
        .whenMatchedUpdate(
            set = {
                "tgt.MoxieCaseNumber": "src.MoxieCaseNumber",
                "tgt.SalesOrderNumber": "src.SalesOrderNumber",
                "tgt.UpdatedDate": current_timestamp(),
                "tgt.UpdatedBy": lit(CreatedBy)
                }
        )
        .execute()
        )

        #Creating Log Entry
        log_dict = {
        "ConfigID" : ConfigID,
        "SourceTypeID" : sourceTypeId,
        "Source" : "rawzone.",
        "SourceFileName" : "",
        "Destination" : "promotion.",
        "DestinationFileName" : "",
        "Run_ID": str(batchId),
        "Job_ID": str(JobId)
        }
        df_audit = spark.createDataFrame([log_dict])

        logIntoPromotionLogTable(df_audit,CreatedBy,"Succeeded")
    except Exception as e:
        logIntoPromotionLogTable(df_audit,CreatedBy,"Failed",str(e))
        raise e

In [0]:
q=(df_Source.writeStream
                  .format("delta")
                  .queryName("Raw_BillingCase")
                  .trigger(availableNow=True)
                  .foreachBatch(upsertToDelta)
                  .option("checkpointLocation", silverpath_InvoiceAddendum+checkPointLocation)
                  .outputMode("update")
                  .start()
)